In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.9 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import SimpleITK as sitk

# =============================================================================
# WITHIN-BRAIN Z-SCORE NORMALIZATION
#
# Why this script exists:
#   The N4 step does not put intensities on a common scale. MRI is in arbitrary
#   units, so the same tumor looks like ~370 on one case and ~52000 on another.
#   With fixed bin SIZE (binWidth) that makes the number of gray levels (Ng)
#   vary wildly per case, which corrupts texture features and (at the extreme)
#   hangs GLCM. The fix is to standardize intensities BEFORE extraction.
#
# What it does:
#   For every case, for every sequence, it reads the existing N4 file, computes
#   mean and std over the BRAIN voxels only (voxels > 0, since the images are
#   skull-stripped and background is exactly 0), z-scores the brain, and leaves
#   the background at 0. Each sequence of each case is normalized by its OWN
#   brain statistics. Output is written as <seq>_norm.nii.gz next to the N4 file.
#
# Notes:
#   - This reuses the N4 output, so N4 is NOT re-run (cheap).
#   - Background is held at exactly 0 so LoG/Wavelet filters see a neutral
#     value outside the brain instead of a large negative constant.
#   - Brain stats include the tumor voxels, which is the standard within-subject
#     z-score convention; the tumor is a small fraction of the brain volume.
# =============================================================================

# --- CONFIG: edit this path to wherever your preprocessed folders live ---
PREPROCESSED_DIR = '/content/drive/MyDrive/bsd/11_radiomics_for_gliomas/02_train/trial_80/preprocessed'

sequences_to_normalize = ['flair', 't1ce', 't1', 't2']

# Set True to normalize only ONE case and print its stats (no other files touched).
SMOKE_TEST = False
SMOKE_CASE = 'BT0791'

# --- Collect case folders (directories only, skip helper folders like _qc_overlays) ---
all_entries = sorted(os.listdir(PREPROCESSED_DIR))
case_folders = []
for entry in all_entries:
    entry_path = os.path.join(PREPROCESSED_DIR, entry)
    if not os.path.isdir(entry_path):
        continue
    if entry.startswith('_'):
        continue
    case_folders.append(entry)

if SMOKE_TEST:
    case_folders = [SMOKE_CASE]
    print(f"SMOKE TEST MODE: normalizing only {SMOKE_CASE}\n")
else:
    print(f"Found {len(case_folders)} case folders to normalize.\n")

# --- Main loop: case, then sequence ---
for case_folder in case_folders:

    case_path = os.path.join(PREPROCESSED_DIR, case_folder)
    print(f"Normalizing: {case_folder}")

    for seq in sequences_to_normalize:

        # Find the N4 file for this sequence.
        # Handles '<seq>_n4.nii.gz' (TCGA / UCSF / UTSW) and
        # '<seq>_n4_stripped.nii.gz' (EGD) via the startswith match.
        n4_file_path = None
        for file_name in os.listdir(case_path):
            if file_name.startswith(f"{seq}_n4") and file_name.endswith('.nii.gz'):
                n4_file_path = os.path.join(case_path, file_name)
                break

        if n4_file_path is None:
            print(f"  -> WARNING: no N4 file for {seq}, skipping.")
            continue

        # Read the N4-corrected image
        sitk_img = sitk.ReadImage(n4_file_path)
        img_array = sitk.GetArrayFromImage(sitk_img).astype(np.float32)

        # Brain = strictly positive voxels (skull-stripped, background is 0)
        brain_mask = img_array > 0
        n_brain_voxels = int(brain_mask.sum())

        if n_brain_voxels == 0:
            print(f"  -> WARNING: {seq} has no positive voxels, skipping.")
            continue

        # Mean and std over brain voxels only
        brain_values = img_array[brain_mask]
        brain_mean = float(brain_values.mean())
        brain_std = float(brain_values.std())

        if brain_std == 0:
            print(f"  -> WARNING: {seq} brain std is 0, skipping.")
            continue

        # Z-score the brain; leave background at 0
        normalized_array = np.zeros_like(img_array, dtype=np.float32)
        normalized_array[brain_mask] = (img_array[brain_mask] - brain_mean) / brain_std

        # Build the output image and copy geometry (spacing/origin/direction) from the N4 image
        normalized_sitk = sitk.GetImageFromArray(normalized_array)
        normalized_sitk.CopyInformation(sitk_img)

        out_file_path = os.path.join(case_path, f"{seq}_norm.nii.gz")
        sitk.WriteImage(normalized_sitk, out_file_path)

        print(f"  -> {seq}: brain mean {brain_mean:.2f}, std {brain_std:.2f} -> saved {seq}_norm.nii.gz")

    print("")

print("All cases normalized." if not SMOKE_TEST else "Smoke test done.")

Found 2061 case folders to normalize.

Normalizing: BT0001
  -> flair: brain mean 378.14, std 105.09 -> saved flair_norm.nii.gz
  -> t1ce: brain mean 512.20, std 97.23 -> saved t1ce_norm.nii.gz
  -> t1: brain mean 508.71, std 88.65 -> saved t1_norm.nii.gz
  -> t2: brain mean 481.83, std 181.37 -> saved t2_norm.nii.gz

Normalizing: BT0002
  -> flair: brain mean 375.11, std 83.43 -> saved flair_norm.nii.gz
  -> t1ce: brain mean 1163.37, std 179.14 -> saved t1ce_norm.nii.gz
  -> t1: brain mean 1205.71, std 195.54 -> saved t1_norm.nii.gz
  -> t2: brain mean 683.44, std 217.99 -> saved t2_norm.nii.gz

Normalizing: BT0003
  -> flair: brain mean 560.04, std 121.55 -> saved flair_norm.nii.gz
  -> t1ce: brain mean 1442.34, std 256.73 -> saved t1ce_norm.nii.gz
  -> t1: brain mean 1646.92, std 279.02 -> saved t1_norm.nii.gz
  -> t2: brain mean 931.71, std 367.53 -> saved t2_norm.nii.gz

Normalizing: BT0005
  -> flair: brain mean 511.82, std 146.68 -> saved flair_norm.nii.gz
  -> t1ce: brain mean 